In [7]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cholesky
import os
import sys

sys.path.append(os.path.abspath(".."))  # Adds the parent directory to sys.path

from privateggm.model import optimize_lbfgs, mirror_descent
from privateggm.utils import *

In [8]:
# Set seed for reproducibility
np.random.seed(42)

# generate a toy 3x3 covariance example
def generate_toy_example(Sigma_true, inds, noise_scale):
    """
    Generate a small 3x3 synthetic covariance matrix and parameters for testing.
    """

    # generate symmetric noise matrix
    lower_triangle_noise = np.tril(np.random.normal(0, noise_scale, Sigma_true.shape))
    tau = lower_triangle_noise + lower_triangle_noise.T - np.diag(np.diag(lower_triangle_noise))
    # perturb Sigma_true
    Sigma_tilde = Sigma_true + tau

    # select indices of the observed entries (only diagonal and lower triangle indices)
    rows, cols = inds[:, 0], inds[:, 1]
    Sigma_tilde_vec = Sigma_tilde[rows, cols]

    # Show mask of observed entries
    Sigma_mask = np.zeros_like(Sigma_tilde)
    for i, j in inds:
        Sigma_mask[i, j] = 1
        Sigma_mask[j, i] = 1  # Ensure symmetry if needed
    print(f"Observed entries mask:\n{Sigma_mask}")

    z_init = initialize_z(Sigma_tilde, inds)

    # For mirror descent
    Sigma_init = np.eye(3) 

    return  Sigma_tilde, Sigma_tilde_vec, z_init, Sigma_init, tau

def test_lbfgs(Sigma_true, inds, noise_scale, max_iters=1000, verbose=False):
    """
    Run the L-BFGS optimization on the toy example.
    """
    Sigma_tilde, Sigma_tilde_vec, z_init, _, _ = generate_toy_example(Sigma_true, inds, noise_scale)
    
    d = Sigma_true.shape[0]  # Dimensionality
    
    # Run lbfgs optimization
    z_opt, g_vals, obj_vals = optimize_lbfgs(Sigma_tilde_vec, z_init, inds, d, max_iters=max_iters, verbose=verbose)
    
    # Convert optimized z into a precision matrix K
    K_z_opt = get_K(z_opt, inds, d)
    
    # Convert back to covariance matrix
    Sigma_opt = np.linalg.inv(K_z_opt)
    
    # Print results
    print(f"\nTrue Covariance Matrix Sigma:\n{Sigma_true}")
    print(f"\nNoisy Covariance Matrix Sigma:\n{Sigma_tilde}")
    print(f"\nOptimized Covariance Matrix Sigma:\n{Sigma_opt}")
    
    # Compute total MSE
    mse_total = np.mean((Sigma_true - Sigma_opt) ** 2)
    
    # Compute MSE only over observed entries
    observed_errors = (Sigma_true[inds[:, 0], inds[:, 1]] - Sigma_opt[inds[:, 0], inds[:, 1]]) ** 2
    mse_observed = np.mean(observed_errors)
    
    # Print error metrics
    print(f"\nMSE (total) between true and optimized Sigma: {mse_total:.6f}")
    print(f"\nMSE (observed entries only) between true and optimized Sigma: {mse_observed:.6f}")

def test_mirror_descent(Sigma_true, inds, noise_scale, max_iters=20000, eta_init=0.2, tol=1e-8, verbose=True):

    # Generate toy dataset
    Sigma_tilde, _, _, Sigma_init, tau = generate_toy_example(Sigma_true, inds, noise_scale)
    
    # Run Mirror Descent
    Sigma_opt, K_opt = mirror_descent(Sigma_init, Sigma_tilde, inds, tau, 
                                      max_iters=max_iters, eta_init=eta_init, tol=tol, verbose=verbose)
    
    # Print results
    print(f"\nTrue Covariance Matrix Sigma:\n{Sigma_true}")
    print(f"\nNoisy Covariance Matrix Sigma:\n{Sigma_tilde}")
    print(f"\nOptimized Covariance Matrix Sigma (Mirror Descent):\n{Sigma_opt}")
    
    # Compute total MSE
    mse_total = np.mean((Sigma_true - Sigma_opt) ** 2)
    
    # Compute MSE only over observed entries
    observed_errors = (Sigma_true[inds[:, 0], inds[:, 1]] - Sigma_opt[inds[:, 0], inds[:, 1]]) ** 2
    mse_observed = np.mean(observed_errors)
    
    # Print error metrics
    print(f"\nMSE (total) between true and optimized Sigma: {mse_total:.6f}")
    print(f"\nMSE (observed entries only) between true and optimized Sigma: {mse_observed:.6f}")


In [17]:
np.random.seed(42)

# Define a well-conditioned covariance matrix
d = 3
Sigma_true = np.array([[2.0, 0.5, 0.3],
                       [0.5, 1.5, 0.4],
                       [0.3, 0.4, 1.0]])
noise_scale = 0.1

inds = np.array([[0,0], [1,1], [2,2], [1,0], [2,0]])

test_lbfgs(Sigma_true, inds, noise_scale, max_iters=2000, verbose=True)
print("-"*50)
test_mirror_descent(Sigma_true, inds, noise_scale, max_iters=2000, eta_init=0.01, tol=1e-6, verbose=True)

Observed entries mask:
[[1. 1. 1.]
 [1. 1. 0.]
 [1. 0. 1.]]
Iteration 0: Loss = 4.059326e+00, Grad Sum = 2.220449e+00, Step Size = 1.000000e+00
Iteration 1: Loss = 3.862610e+00, Grad Sum = 1.445340e+00, Step Size = 1.000000e+00
Iteration 2: Loss = 3.804332e+00, Grad Sum = 7.928121e-01, Step Size = 1.000000e+00
Iteration 3: Loss = 3.796853e+00, Grad Sum = 2.238699e-01, Step Size = 1.000000e+00
Iteration 4: Loss = 3.795656e+00, Grad Sum = 1.617817e-01, Step Size = 1.000000e+00
Iteration 5: Loss = 3.794361e+00, Grad Sum = 8.152925e-02, Step Size = 1.000000e+00
Iteration 6: Loss = 3.794288e+00, Grad Sum = 4.017826e-02, Step Size = 1.000000e+00
Iteration 7: Loss = 3.794263e+00, Grad Sum = 7.769105e-03, Step Size = 1.000000e+00
Iteration 8: Loss = 3.794262e+00, Grad Sum = 4.146353e-03, Step Size = 1.000000e+00
Iteration 9: Loss = 3.794261e+00, Grad Sum = 2.837188e-04, Step Size = 1.000000e+00
Optimization converged at iteration 9: Loss Change = 6.2042490e-07

True Covariance Matrix Sigma:
[[